# Data Description
- Background: 候選人社群入口的原始公開網頁、候選人資料 JS 與 HTTP 存取紀錄。
- date received: 20261019
- Path to data file: 來源 URL 見 capture_manifest；實際檔案位於 data/raw/round3_20261019/。
- Unit of observation: 一次已記錄的來源 URL 取得結果。
- Sample period: 2026-10-19 擷取快照；來源內容可能早於本日。
- Known issues: 403、000 與頁框不是貼文。未重試已失敗來源；不以 HTTP 200 證明社群內容可用。聯盟頁可能包含研究名冊以外的其他層級參選人。
- Definition for each variable:
  - url、kind、linked_from：來源、來源用途與發現頁。
  - captured_date：擷取日；http_status、return_code、error：實際 HTTP／curl 狀態。
  - path、sha256：保存 bytes 的專案相對路徑與 SHA-256；無檔案者留空。
- Data path: data/raw/round3_20261019/capture_manifest.json，及逐筆 path 列示的原始檔。
- Role: raw
- Verification status: 以下程式核對固定 manifest 與所有已保存原始檔 hash，列出實際樣本與狀態分布；不變更來源。
- SHA-256: 固定值見程式。
- Next processing notebook: data/code/prepare_social_accounts_20261019.ipynb


In [1]:
from pathlib import Path
import pandas as pd,json,hashlib
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()


## Change for Your Own Data
固定來源版本如下；正常重跑不改預期 hash。

In [2]:
assert sha(ROOT/'data/raw/round3_20261019/capture_manifest.json')=='a701143ad22134673e670515332b9691ba9d3f911c9903671e290079509de24f'
records=json.loads((ROOT/'data/raw/round3_20261019/capture_manifest.json').read_text())
for r in records:
 if r['path']: assert sha(ROOT/r['path'])==r['sha256']
df=pd.DataFrame(records)


## Summary

In [3]:
print({'urls':len(df),'saved_files':int(df.path.ne('').sum())});print(df.http_status.value_counts().to_string())

{'urls': 48, 'saved_files': 45}
http_status
200    43
000     3
403     2


### Sample

In [4]:
display(df.head(10))

,url,kind,linked_from,captured_date,http_status,return_code,error,path,sha256
0,https://teamtaiwan.dpp.org.tw/,party_home,,2026-10-19,200,0,,data/raw/round3_20261019/aa62505c97df5dccab03....,61984cbe7b23bfe0bb2a45eaff9f5a8f9c142a036a5ce5...
1,https://teamtaiwan.dpp.org.tw/asset/types/elec...,linked_candidate_data,https://teamtaiwan.dpp.org.tw/,2026-10-19,200,0,,data/raw/round3_20261019/cab91a155008312e422c....,30ea7e9296a285fd11a374250c7735892c0ba965a3d59e...
2,https://teamtaiwan.dpp.org.tw/councilor,party_candidates,,2026-10-19,200,0,,data/raw/round3_20261019/192f32811d970a8d4667....,f8d05cdf6d3cca2cb697a4c290bb9666cfde5bdf10523b...
3,https://www.tpp.org.tw/2026election/,party_candidates,,2026-10-19,403,0,,data/raw/round3_20261019/ae35c188492559f15f4a....,9a7f79d4e483bed5911b2bb28956e940e34ee0122a5f93...
4,https://www.tpp.org.tw/election2026/candidates...,party_candidates,,2026-10-19,403,0,,data/raw/round3_20261019/29a36c32638cc045b32e....,9e7ff9739e7569e9ad169c0f1fd69e028d97050bff6de1...
5,https://2026.kmtks.org.tw/,party_candidates,,2026-10-19,000,35,curl: (35) Recv failure: Connection reset by peer,,
6,https://www1.kmt.org.tw/page.aspx?mid=40,party_candidates,,2026-10-19,200,0,,data/raw/round3_20261019/d3dac1637ec6014659c9....,acbfd8cb0afa3fa2c419ea44b81a4bddbf4e51e776cc22...
7,https://www.taiwangogo.tw/candidates/,alliance_candidates,,2026-10-19,200,0,,data/raw/round3_20261019/c48fcb12f32e5588c2ca....,e4bd203d8358d8528e6fa5b86aa81660096e50a9d2334e...
8,https://www.taiwangogo.tw/candidates/#,alliance_candidate_profile,https://www.taiwangogo.tw/candidates/,2026-10-19,200,0,,data/raw/round3_20261019/2c0f4a78ae268055c1a6....,8a4f5c5c4b3698aa0b41ac163774fb1de4db456186051c...
9,https://www.taiwangogo.tw/hsieh-min-hung/,alliance_candidate_profile,https://www.taiwangogo.tw/candidates/,2026-10-19,200,0,,data/raw/round3_20261019/50b22ce8b7a3e6fd7409....,7b6f03fafac81cc5a11e9cf57b85912efee9bc2c304fff...


### Summary Stats

In [5]:
display(df[['kind','http_status','return_code']].describe(include='all'))

,kind,http_status,return_code
count,48,48,48
unique,9,3,3
top,alliance_candidate_profile,200,0
freq,36,43,45
